# 04 Bronze to Silver
MERGEs the staged batch into Silver (current state per `customer_id`). If a customer changed more than once in the batch, only the latest version is merged, because Delta rejects a MERGE where several source rows match one target row.

In [ ]:
%sql
MERGE INTO silver.sales s
USING (
    SELECT customer_id, amount, region, updated_at
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
        FROM staging.sales_incremental
    )
    WHERE rn = 1
) b
ON s.customer_id = b.customer_id
WHEN MATCHED THEN UPDATE SET
    s.amount = b.amount,
    s.region = b.region,
    s.updated_at = b.updated_at
WHEN NOT MATCHED THEN INSERT (customer_id, amount, region, updated_at)
VALUES (b.customer_id, b.amount, b.region, b.updated_at)

In [ ]:
# Row counts of the MERGE (inserted / updated) from the Delta history.
metrics = spark.sql("DESCRIBE HISTORY silver.sales LIMIT 1").select("operationMetrics").collect()[0][0]
print({k: metrics[k] for k in ("numSourceRows", "numTargetRowsInserted", "numTargetRowsUpdated") if k in metrics})